# AgroScan — Jalon 5 : renforcement des classes faibles

Objectif : partir du dernier modèle mobile (`jalon4_original_segmented_plantdoc_ft.keras`) et tester deux pistes mentionnées dans les perspectives du rapport :

1. pondération de loss sur les classes faibles ;
2. sur-échantillonnage + augmentation ciblée des classes faibles.

Le notebook garde la même logique que le jalon 4 : évaluation PlantDoc original et PlantDoc segmenté, métriques top-k, macro F1, et focus sur les classes faibles.


## 1. Lancement

Linux avec GPU TensorFlow :

```bash
./scripts/tf_gpu_env.sh jupyter lab
```

Le notebook est prévu pour être lancé depuis la racine du projet ou depuis `notebooks/`.


## 2. Configuration


In [15]:
from pathlib import Path
import os
import json
import math
import random
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
import tensorflow_datasets as tfds

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

# Détection robuste de la racine projet.
_env_root = Path(os.environ['AGROSCAN_ROOT']) if 'AGROSCAN_ROOT' in os.environ else None
_cwd = Path('.').resolve()
try:
    _starting_dir = Path(get_ipython().starting_dir).resolve()
except Exception:
    _starting_dir = _cwd

candidates = []
if _env_root is not None:
    candidates.append(_env_root)
for base in [_cwd, _starting_dir, *_cwd.parents, *_starting_dir.parents]:
    candidates.append(base)

PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'README.md').exists() and (candidate / 'notebooks').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Impossible de trouver la racine du projet AgroScan.')

DATA_DIR = PROJECT_ROOT / 'data'
MODEL_DIR = PROJECT_ROOT / 'models'
REPORT_DIR = PROJECT_ROOT / 'reports' / 'jalon5_weak_classes'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

VAL_RATIO = 0.2
WEAK_F1_THRESHOLD = 0.45
MIN_WEAK_CLASSES = 5
MAX_CLASS_WEIGHT = 4.0
WEAK_WEIGHT_MULTIPLIER = 1.35
WEAK_SAMPLE_WEIGHT = 0.40
COMBINED_WEIGHT_BLEND = 0.50

LR_WEIGHTED = 1e-5
LR_TARGETED = 8e-6
LR_COMBINED = 6e-6
EPOCHS_WEIGHTED = 12
EPOCHS_TARGETED = 12
EPOCHS_COMBINED = 10
PATIENCE = 4
UNFREEZE_RATIO = 0.25

print('Project root:', PROJECT_ROOT)
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


Project root: /home/leonard/UQAC/8INF934 - Atelier Pratique IA I/Projet/AgroScan
TensorFlow: 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 3. Chargement PlantVillage et mapping PlantDoc


In [2]:
(_, _, pv_test_raw), info = tfds.load(
    'plant_village',
    split=['train[:70%]', 'train[70%:85%]', 'train[85%:]'],
    as_supervised=True,
    with_info=True,
    data_dir=str(DATA_DIR / 'tfds'),
)

class_names = info.features['label'].names
num_classes = len(class_names)
pv_label_to_idx = {name: i for i, name in enumerate(class_names)}
idx_to_label = {i: name for i, name in enumerate(class_names)}
print(f'PlantVillage labels: {num_classes}')


PlantVillage labels: 38


W0000 00:00:1781019351.377030   11349 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


In [3]:
PLANTDOC_DIR = DATA_DIR / 'plantdoc'
PLANTDOC_TRAIN_DIR = PLANTDOC_DIR / 'train'
PLANTDOC_TEST_DIR = PLANTDOC_DIR / 'test'
SEGMENTED_PLANTDOC_DIR = DATA_DIR / 'segmented' / 'plantdoc_rembg_crop_gray'
SEGMENTED_PLANTDOC_TRAIN_DIR = SEGMENTED_PLANTDOC_DIR / 'train'
SEGMENTED_PLANTDOC_TEST_DIR = SEGMENTED_PLANTDOC_DIR / 'test'

plantdoc_to_plantvillage = {
    'Apple Scab Leaf':                       'Apple___Apple_scab',
    'Apple leaf':                            'Apple___healthy',
    'Apple rust leaf':                       'Apple___Cedar_apple_rust',
    'Bell_pepper leaf':                      'Pepper,_bell___healthy',
    'Bell_pepper leaf spot':                 'Pepper,_bell___Bacterial_spot',
    'Blueberry leaf':                        'Blueberry___healthy',
    'Cherry leaf':                           'Cherry___healthy',
    'Corn Gray leaf spot':                   'Corn___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn leaf blight':                      'Corn___Northern_Leaf_Blight',
    'Corn rust leaf':                        'Corn___Common_rust',
    'Peach leaf':                            'Peach___healthy',
    'Potato leaf early blight':              'Potato___Early_blight',
    'Potato leaf late blight':               'Potato___Late_blight',
    'Raspberry leaf':                        'Raspberry___healthy',
    'Soyabean leaf':                         'Soybean___healthy',
    'Squash Powdery mildew leaf':            'Squash___Powdery_mildew',
    'Strawberry leaf':                       'Strawberry___healthy',
    'Tomato Early blight leaf':              'Tomato___Early_blight',
    'Tomato Septoria leaf spot':             'Tomato___Septoria_leaf_spot',
    'Tomato leaf':                           'Tomato___healthy',
    'Tomato leaf bacterial spot':            'Tomato___Bacterial_spot',
    'Tomato leaf late blight':               'Tomato___Late_blight',
    'Tomato leaf mosaic virus':              'Tomato___Tomato_mosaic_virus',
    'Tomato leaf yellow virus':              'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato mold leaf':                      'Tomato___Leaf_Mold',
    'Tomato two spotted spider mites leaf':  'Tomato___Spider_mites Two-spotted_spider_mite',
    'grape leaf':                            'Grape___healthy',
    'grape leaf black rot':                  'Grape___Black_rot',
}

if not PLANTDOC_TRAIN_DIR.exists() or not PLANTDOC_TEST_DIR.exists():
    raise FileNotFoundError('PlantDoc introuvable. Attendu : data/plantdoc/train et data/plantdoc/test')
if not SEGMENTED_PLANTDOC_TRAIN_DIR.exists() or not SEGMENTED_PLANTDOC_TEST_DIR.exists():
    raise FileNotFoundError('PlantDoc segmenté introuvable. Lancer d’abord jalon4_segmentation_experiment.ipynb.')

valid_pd_mapping = {
    k: v for k, v in plantdoc_to_plantvillage.items()
    if (PLANTDOC_TRAIN_DIR / k).exists() and (PLANTDOC_TEST_DIR / k).exists() and v in pv_label_to_idx
}
print(f'PlantDoc classes mappées: {len(valid_pd_mapping)}')


PlantDoc classes mappées: 27


## 4. Collecte des chemins et splits


In [4]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def collect_mapped_image_paths(root_dir, mapping):
    paths, labels, source_classes = [], [], []
    for src_cls, pv_cls in mapping.items():
        cls_dir = root_dir / src_cls
        if not cls_dir.exists() or pv_cls not in pv_label_to_idx:
            continue
        label = pv_label_to_idx[pv_cls]
        for p in sorted(cls_dir.iterdir()):
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                paths.append(str(p))
                labels.append(label)
                source_classes.append(src_cls)
    return np.array(paths, dtype=str), np.array(labels, dtype=np.int64), np.array(source_classes, dtype=str)

def stratified_train_val_split(paths, labels, val_ratio=0.2, seed=42):
    rng = np.random.default_rng(seed)
    train_idx, val_idx = [], []
    labels = np.asarray(labels)
    for label in sorted(set(labels.tolist())):
        idx = np.where(labels == label)[0]
        idx = rng.permutation(idx)
        n_val = max(1, int(round(len(idx) * val_ratio))) if len(idx) > 1 else 0
        val_idx.extend(idx[:n_val].tolist())
        train_idx.extend(idx[n_val:].tolist())
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return np.array(train_idx), np.array(val_idx)

orig_train_paths, orig_train_labels, _ = collect_mapped_image_paths(PLANTDOC_TRAIN_DIR, valid_pd_mapping)
orig_test_paths, orig_test_labels, _ = collect_mapped_image_paths(PLANTDOC_TEST_DIR, valid_pd_mapping)
seg_train_paths, seg_train_labels, _ = collect_mapped_image_paths(SEGMENTED_PLANTDOC_TRAIN_DIR, valid_pd_mapping)
seg_test_paths, seg_test_labels, _ = collect_mapped_image_paths(SEGMENTED_PLANTDOC_TEST_DIR, valid_pd_mapping)

orig_train_idx, orig_val_idx = stratified_train_val_split(orig_train_paths, orig_train_labels, VAL_RATIO, SEED)
seg_train_idx, seg_val_idx = stratified_train_val_split(seg_train_paths, seg_train_labels, VAL_RATIO, SEED)

train_paths = np.concatenate([orig_train_paths[orig_train_idx], seg_train_paths[seg_train_idx]])
train_labels = np.concatenate([orig_train_labels[orig_train_idx], seg_train_labels[seg_train_idx]])
val_paths = np.concatenate([orig_train_paths[orig_val_idx], seg_train_paths[seg_val_idx]])
val_labels = np.concatenate([orig_train_labels[orig_val_idx], seg_train_labels[seg_val_idx]])

summary = pd.DataFrame([
    {'split': 'train_original', 'n': len(orig_train_paths[orig_train_idx])},
    {'split': 'val_original', 'n': len(orig_train_paths[orig_val_idx])},
    {'split': 'train_segmented', 'n': len(seg_train_paths[seg_train_idx])},
    {'split': 'val_segmented', 'n': len(seg_train_paths[seg_val_idx])},
    {'split': 'test_original', 'n': len(orig_test_paths)},
    {'split': 'test_segmented', 'n': len(seg_test_paths)},
])
display(summary)


,split,n
0,train_original,1873
1,val_original,467
2,train_segmented,1873
3,val_segmented,467
4,test_original,236
5,test_segmented,236


## 5. Prétraitement, augmentation et datasets


In [5]:
base_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal', seed=SEED),
    tf.keras.layers.RandomRotation(0.08, fill_mode='reflect', seed=SEED),
    tf.keras.layers.RandomZoom(0.12, fill_mode='reflect', seed=SEED),
    tf.keras.layers.RandomContrast(0.12, seed=SEED),
], name='base_augmentation')

strong_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal', seed=SEED + 1),
    tf.keras.layers.RandomRotation(0.14, fill_mode='reflect', seed=SEED + 1),
    tf.keras.layers.RandomZoom(0.18, fill_mode='reflect', seed=SEED + 1),
    tf.keras.layers.RandomContrast(0.20, seed=SEED + 1),
    tf.keras.layers.RandomBrightness(0.10, value_range=(0, 255), seed=SEED + 1),
], name='weak_class_augmentation')


def load_image(path):
    raw = tf.io.read_file(path)
    image = tf.image.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE, method='bilinear')
    image = tf.cast(image, tf.float32)
    return image


def add_light_noise(image):
    noise = tf.random.normal(tf.shape(image), mean=0.0, stddev=4.0, seed=SEED)
    return tf.clip_by_value(image + noise, 0.0, 255.0)


def preprocess_base(path, label, training=False):
    image = load_image(path)
    if training:
        image = base_augmentation(image, training=True)
    return image, tf.cast(label, tf.int64)


def preprocess_strong(path, label):
    image = load_image(path)
    image = strong_augmentation(image, training=True)
    image = add_light_noise(image)
    return image, tf.cast(label, tf.int64)


def make_dataset(paths, labels, training=False, batch_size=BATCH_SIZE, strong=False, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths.astype(str), labels.astype(np.int64)))
    if training and shuffle:
        ds = ds.shuffle(min(len(paths), 4096), seed=SEED, reshuffle_each_iteration=True)
    if strong:
        ds = ds.map(preprocess_strong, num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(lambda p, y: preprocess_base(p, y, training=training), num_parallel_calls=AUTOTUNE)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

val_ds = make_dataset(val_paths, val_labels, training=False, shuffle=False)
orig_test_ds = make_dataset(orig_test_paths, orig_test_labels, training=False, shuffle=False)
seg_test_ds = make_dataset(seg_test_paths, seg_test_labels, training=False, shuffle=False)


## 6. Fonctions d’évaluation


In [6]:
def predict_dataset(model, ds):
    y_true, probs = [], []
    for images, labels in ds:
        batch_probs = model.predict(images, verbose=0)
        if isinstance(batch_probs, dict):
            batch_probs = batch_probs.get('disease', next(iter(batch_probs.values())))
        y_true.extend(labels.numpy().astype(int).tolist())
        probs.append(np.asarray(batch_probs))
    probs = np.concatenate(probs, axis=0)
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.argmax(probs, axis=1).astype(np.int64)
    conf = np.max(probs, axis=1)
    return y_true, y_pred, probs, conf


def topk_accuracy(y_true, probs, k):
    topk = np.argsort(probs, axis=1)[:, -k:]
    return float(np.mean([y in row for y, row in zip(y_true, topk)]))


def metrics_from_predictions(y_true, y_pred, probs, weak_indices=None):
    row = {
        'top1_accuracy': accuracy_score(y_true, y_pred),
        'top2_accuracy': topk_accuracy(y_true, probs, 2),
        'top3_accuracy': topk_accuracy(y_true, probs, 3),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'mean_confidence': float(np.max(probs, axis=1).mean()),
        'n': int(len(y_true)),
    }
    if weak_indices is not None and len(weak_indices) > 0:
        mask = np.isin(y_true, list(weak_indices))
        row['weak_n'] = int(mask.sum())
        row['weak_top1_accuracy'] = accuracy_score(y_true[mask], y_pred[mask]) if mask.any() else np.nan
        row['weak_macro_f1'] = f1_score(y_true[mask], y_pred[mask], average='macro', zero_division=0) if mask.any() else np.nan
    return row


def evaluate_model(model, name, weak_indices=None):
    rows = []
    for split_name, ds in [('plantdoc_original', orig_test_ds), ('plantdoc_segmented', seg_test_ds)]:
        y_true, y_pred, probs, conf = predict_dataset(model, ds)
        rows.append({'model': name, 'split': split_name, **metrics_from_predictions(y_true, y_pred, probs, weak_indices)})
        report = pd.DataFrame(classification_report(
            y_true, y_pred,
            labels=sorted(set(y_true.tolist()) | set(y_pred.tolist())),
            target_names=[class_names[i] for i in sorted(set(y_true.tolist()) | set(y_pred.tolist()))],
            output_dict=True,
            zero_division=0,
        )).T
        report.to_csv(REPORT_DIR / f'{name}_{split_name}_classification_report.csv')
    return pd.DataFrame(rows)


## 7. Baseline : dernier modèle Jalon 4


In [7]:
MODEL_CANDIDATES = [
    MODEL_DIR / 'jalon4_original_segmented_plantdoc_ft.keras',
    MODEL_DIR / 'jalon4_multidomain_balanced.keras',
    MODEL_DIR / 'agroscan_jalon4.keras',
]
model_path = next((p for p in MODEL_CANDIDATES if p.exists()), None)
if model_path is None:
    raise FileNotFoundError('Aucun modèle de départ trouvé. Attendu : models/jalon4_original_segmented_plantdoc_ft.keras')
print('Modèle de départ:', model_path)

baseline_model = tf.keras.models.load_model(model_path, compile=False)
baseline_rows = evaluate_model(baseline_model, 'baseline_jalon4')
display(baseline_rows)
baseline_rows.to_csv(REPORT_DIR / 'baseline_metrics.csv', index=False)


Modèle de départ: /home/leonard/UQAC/8INF934 - Atelier Pratique IA I/Projet/AgroScan/models/jalon4_original_segmented_plantdoc_ft.keras


E0000 00:00:1781019369.596978   11349 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


,model,split,top1_accuracy,top2_accuracy,top3_accuracy,macro_f1,mean_confidence,n
0,baseline_jalon4,plantdoc_original,0.516949,0.673729,0.775424,0.501717,0.578748,236
1,baseline_jalon4,plantdoc_segmented,0.508475,0.677966,0.758475,0.452295,0.552088,236


## 8. Identification des classes faibles


In [8]:
y_true_base, y_pred_base, probs_base, conf_base = predict_dataset(baseline_model, orig_test_ds)
labels_present = sorted(set(y_true_base.tolist()) | set(y_pred_base.tolist()))
base_report = pd.DataFrame(classification_report(
    y_true_base,
    y_pred_base,
    labels=labels_present,
    target_names=[class_names[i] for i in labels_present],
    output_dict=True,
    zero_division=0,
)).T
base_report.to_csv(REPORT_DIR / 'baseline_original_classification_report.csv')

class_rows = []
for idx in sorted(set(y_true_base.tolist())):
    label_name = class_names[idx]
    if label_name not in base_report.index:
        continue
    row = base_report.loc[label_name]
    class_rows.append({
        'class_idx': idx,
        'class_name': label_name,
        'support': int(row['support']),
        'precision': float(row['precision']),
        'recall': float(row['recall']),
        'f1': float(row['f1-score']),
    })
class_perf = pd.DataFrame(class_rows).sort_values(['f1', 'support'], ascending=[True, False]).reset_index(drop=True)
weak_df = class_perf[class_perf['f1'] < WEAK_F1_THRESHOLD].copy()
if len(weak_df) < MIN_WEAK_CLASSES:
    weak_df = class_perf.head(MIN_WEAK_CLASSES).copy()
weak_indices = set(weak_df['class_idx'].astype(int).tolist())

print(f'Classes faibles retenues: {len(weak_indices)}')
display(weak_df)
weak_df.to_csv(REPORT_DIR / 'weak_classes_selected.csv', index=False)


Classes faibles retenues: 14


E0000 00:00:1781019385.179092   11349 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


,class_idx,class_name,support,precision,recall,f1
0,28,Tomato___Bacterial_spot,9,0.111111,0.111111,0.111111
1,22,Potato___Late_blight,8,0.200000,0.125000,0.153846
2,36,Tomato___Tomato_mosaic_virus,10,0.500000,0.100000,0.166667
3,32,Tomato___Leaf_Mold,6,0.500000,0.166667,0.250000
4,17,Peach___healthy,9,0.666667,0.222222,0.333333
5,29,Tomato___Early_blight,9,0.375000,0.333333,0.352941
6,33,Tomato___Septoria_leaf_spot,11,0.400000,0.363636,0.380952
7,2,Apple___Cedar_apple_rust,10,0.333333,0.500000,0.400000
8,20,Potato___Early_blight,8,0.294118,0.625000,0.400000
9,7,Corn___Cercospora_leaf_spot Gray_leaf_spot,4,0.333333,0.500000,0.400000


## 9. Préparation class weights et datasets ciblés


In [17]:
# Pondération globale basée sur les labels de train, puis boost limité sur les classes faibles.
present_classes = np.unique(train_labels)
weights = compute_class_weight(class_weight='balanced', classes=present_classes, y=train_labels)
class_weight = {int(c): float(min(w, MAX_CLASS_WEIGHT)) for c, w in zip(present_classes, weights)}
for idx in weak_indices:
    class_weight[int(idx)] = float(min(class_weight.get(int(idx), 1.0) * WEAK_WEIGHT_MULTIPLIER, MAX_CLASS_WEIGHT))


# Variante combinée légère : on rapproche les poids de 1.0 pour éviter de sur-corriger
# les classes faibles déjà sur-échantillonnées.
combined_class_weight = {
    int(cls): float(1.0 + COMBINED_WEIGHT_BLEND * (weight - 1.0))
    for cls, weight in class_weight.items()
}

weights_df = pd.DataFrame([
    {'class_idx': i, 'class_name': class_names[i], 'weight': class_weight.get(i, 1.0), 'combined_weight': combined_class_weight.get(i, 1.0), 'is_weak': i in weak_indices}
    for i in sorted(present_classes.tolist())
]).sort_values(['is_weak', 'weight'], ascending=[False, False])
display(weights_df.head(20))
weights_df.to_csv(REPORT_DIR / 'class_weights.csv', index=False)

train_ds = make_dataset(train_paths, train_labels, training=True, strong=False)

weak_mask_train = np.isin(train_labels, list(weak_indices))
weak_train_paths = train_paths[weak_mask_train]
weak_train_labels = train_labels[weak_mask_train]
print('Train total:', len(train_paths), 'Weak train:', len(weak_train_paths))

base_unbatched = tf.data.Dataset.from_tensor_slices((train_paths.astype(str), train_labels.astype(np.int64)))
base_unbatched = base_unbatched.shuffle(min(len(train_paths), 4096), seed=SEED, reshuffle_each_iteration=True)
base_unbatched = base_unbatched.map(lambda p, y: preprocess_base(p, y, training=True), num_parallel_calls=AUTOTUNE)

weak_unbatched = tf.data.Dataset.from_tensor_slices((weak_train_paths.astype(str), weak_train_labels.astype(np.int64)))
weak_unbatched = weak_unbatched.shuffle(max(len(weak_train_paths), 1), seed=SEED + 7, reshuffle_each_iteration=True)
weak_unbatched = weak_unbatched.map(preprocess_strong, num_parallel_calls=AUTOTUNE)

mixed_targeted_ds = tf.data.Dataset.sample_from_datasets(
    [base_unbatched.repeat(), weak_unbatched.repeat()],
    weights=[1.0 - WEAK_SAMPLE_WEIGHT, WEAK_SAMPLE_WEIGHT],
    seed=SEED,
).batch(BATCH_SIZE).prefetch(AUTOTUNE)

steps_per_epoch = math.ceil(len(train_paths) / BATCH_SIZE)
print('steps_per_epoch:', steps_per_epoch)


,class_idx,class_name,weight,combined_weight,is_weak
25,36,Tomato___Tomato_mosaic_virus,2.675714,1.837857,True
4,5,Cherry___healthy,2.464474,1.732237,True
21,30,Tomato___healthy,2.128409,1.564205,True
5,7,Corn___Cercospora_leaf_spot Gray_leaf_spot,1.836275,1.418137,True
1,2,Apple___Cedar_apple_rust,1.486508,1.243254,True
20,29,Tomato___Early_blight,1.486508,1.243254,True
0,0,Apple___Apple_scab,1.418939,1.209470,True
2,3,Apple___healthy,1.418939,1.209470,True
23,32,Tomato___Leaf_Mold,1.377206,1.188603,True
14,22,Potato___Late_blight,1.200641,1.100321,True


Train total: 3746 Weak train: 1868
steps_per_epoch: 118


## 10. Helpers fine-tuning


In [10]:
def set_partial_unfreeze(model, ratio=0.25):
    backbone = None
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model) and 'MobileNet' in layer.name:
            backbone = layer
            break
    if backbone is None:
        nested_models = [layer for layer in model.layers if isinstance(layer, tf.keras.Model)]
        backbone = nested_models[0] if nested_models else model

    layers = backbone.layers if hasattr(backbone, 'layers') else model.layers
    n_total = len(layers)
    n_train = max(1, int(round(n_total * ratio)))
    cutoff = n_total - n_train
    for i, layer in enumerate(layers):
        layer.trainable = i >= cutoff
    return backbone.name, n_train, n_total


def compile_for_sparse(model, lr):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )


def callbacks_for(name):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            str(MODEL_DIR / f'{name}.keras'),
            monitor='val_accuracy',
            mode='max',
            save_best_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            mode='max',
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.CSVLogger(str(REPORT_DIR / f'{name}_history.csv')),
    ]


## 11. Expérience A — pondération de loss


In [11]:
weighted_model = tf.keras.models.load_model(model_path, compile=False)
backbone_name, n_train, n_total = set_partial_unfreeze(weighted_model, UNFREEZE_RATIO)
print(f'Backbone ciblé: {backbone_name} — couches entraînables: {n_train}/{n_total}')
compile_for_sparse(weighted_model, LR_WEIGHTED)

history_weighted = weighted_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_WEIGHTED,
    class_weight=class_weight,
    callbacks=callbacks_for('jalon5_weighted_loss_ft'),
)

weighted_model.save(MODEL_DIR / 'jalon5_weighted_loss_ft_last.keras')


Backbone ciblé: MobileNetV3Small — couches entraînables: 39/157
Epoch 1/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - accuracy: 0.5001 - loss: 2.1111
Epoch 1: val_accuracy improved from None to 0.52998, saving model to /home/leonard/UQAC/8INF934 - Atelier Pratique IA I/Projet/AgroScan/models/jalon5_weighted_loss_ft.keras

Epoch 1: finished saving model to /home/leonard/UQAC/8INF934 - Atelier Pratique IA I/Projet/AgroScan/models/jalon5_weighted_loss_ft.keras
118/118 ━━━━━━━━━━━━━━━━━━━━ 44s 186ms/step - accuracy: 0.5013 - loss: 2.0461 - val_accuracy: 0.5300 - val_loss: 1.5085
Epoch 2/12
117/118 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step - accuracy: 0.4985 - loss: 1.9939
Epoch 2: val_accuracy improved from 0.52998 to 0.53533, saving model to /home/leonard/UQAC/8INF934 - Atelier Pratique IA I/Projet/AgroScan/models/jalon5_weighted_loss_ft.keras

Epoch 2: finished saving model to /home/leonard/UQAC/8INF934 - Atelier Pratique IA I/Projet/AgroScan/models/jalon5_weighted_loss_ft.keras
118/118 ━━━━━

## 12. Expérience B — sur-échantillonnage + augmentation ciblée


In [12]:
targeted_model = tf.keras.models.load_model(model_path, compile=False)
backbone_name, n_train, n_total = set_partial_unfreeze(targeted_model, UNFREEZE_RATIO)
print(f'Backbone ciblé: {backbone_name} — couches entraînables: {n_train}/{n_total}')
compile_for_sparse(targeted_model, LR_TARGETED)

history_targeted = targeted_model.fit(
    mixed_targeted_ds,
    validation_data=val_ds,
    epochs=EPOCHS_TARGETED,
    steps_per_epoch=steps_per_epoch,
    callbacks=callbacks_for('jalon5_targeted_aug_ft'),
)

targeted_model.save(MODEL_DIR / 'jalon5_targeted_aug_ft_last.keras')


Backbone ciblé: MobileNetV3Small — couches entraînables: 39/157
Epoch 1/12


E0000 00:00:1781019741.640457   11349 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_21}}


118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - accuracy: 0.4214 - loss: 1.8537
Epoch 1: val_accuracy improved from None to 0.53212, saving model to /home/leonard/UQAC/8INF934 - Atelier Pratique IA I/Projet/AgroScan/models/jalon5_targeted_aug_ft.keras

Epoch 1: finished saving model to /home/leonard/UQAC/8INF934 - Atelier Pratique IA I/Projet/AgroScan/models/jalon5_targeted_aug_ft.keras
118/118 ━━━━━━━━━━━━━━━━━━━━ 23s 175ms/step - accuracy: 0.4346 - loss: 1.7863 - val_accuracy: 0.5321 - val_loss: 1.5038
Epoch 2/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.4497 - loss: 1.7157
Epoch 2: val_accuracy did not improve from 0.53212
118/118 ━━━━━━━━━━━━━━━━━━━━ 21s 179ms/step - accuracy: 0.4566 - loss: 1.7059 - val_accuracy: 0.5321 - val_loss: 1.4866
Epoch 3/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step - accuracy: 0.4641 - loss: 1.6686
Epoch 3: val_accuracy improved from 0.53212 to 0.54176, saving model to /home/leonard/UQAC/8INF934 - Atelier Pratique IA I/Projet/AgroScan/models/

## 13. Expérience C — combinaison légère

Cette variante combine le sur-échantillonnage ciblé avec une pondération de loss adoucie. L’objectif est de vérifier si on peut garder le gain du top-k lié aux poids sans trop sur-corriger les classes faibles.


In [18]:
combined_model = tf.keras.models.load_model(model_path, compile=False)
backbone_name, n_train, n_total = set_partial_unfreeze(combined_model, UNFREEZE_RATIO)
print(f'Backbone ciblé: {backbone_name} — couches entraînables: {n_train}/{n_total}')
compile_for_sparse(combined_model, LR_COMBINED)

history_combined = combined_model.fit(
    mixed_targeted_ds,
    validation_data=val_ds,
    epochs=EPOCHS_COMBINED,
    steps_per_epoch=steps_per_epoch,
    class_weight=combined_class_weight,
    callbacks=callbacks_for('jalon5_combined_light_ft'),
)

combined_model.save(MODEL_DIR / 'jalon5_combined_light_ft_last.keras')


Backbone ciblé: MobileNetV3Small — couches entraînables: 39/157
Epoch 1/10


E0000 00:00:1781020518.219250   11349 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_21}}


118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step - accuracy: 0.4334 - loss: 2.1520
Epoch 1: val_accuracy improved from None to 0.52998, saving model to /home/leonard/UQAC/8INF934 - Atelier Pratique IA I/Projet/AgroScan/models/jalon5_combined_light_ft.keras

Epoch 1: finished saving model to /home/leonard/UQAC/8INF934 - Atelier Pratique IA I/Projet/AgroScan/models/jalon5_combined_light_ft.keras
118/118 ━━━━━━━━━━━━━━━━━━━━ 23s 176ms/step - accuracy: 0.4460 - loss: 2.0658 - val_accuracy: 0.5300 - val_loss: 1.5077
Epoch 2/10
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 177ms/step - accuracy: 0.4686 - loss: 1.9899
Epoch 2: val_accuracy improved from 0.52998 to 0.53533, saving model to /home/leonard/UQAC/8INF934 - Atelier Pratique IA I/Projet/AgroScan/models/jalon5_combined_light_ft.keras

Epoch 2: finished saving model to /home/leonard/UQAC/8INF934 - Atelier Pratique IA I/Projet/AgroScan/models/jalon5_combined_light_ft.keras
118/118 ━━━━━━━━━━━━━━━━━━━━ 21s 180ms/step - accuracy: 0.4688 - loss: 1.9896 - val_

## 14. Comparaison finale


In [19]:
# Recharger les meilleurs checkpoints s'ils existent, sinon utiliser les modèles en mémoire.
weighted_best_path = MODEL_DIR / 'jalon5_weighted_loss_ft.keras'
targeted_best_path = MODEL_DIR / 'jalon5_targeted_aug_ft.keras'
combined_best_path = MODEL_DIR / 'jalon5_combined_light_ft.keras'
weighted_eval_model = tf.keras.models.load_model(weighted_best_path if weighted_best_path.exists() else MODEL_DIR / 'jalon5_weighted_loss_ft_last.keras', compile=False)
targeted_eval_model = tf.keras.models.load_model(targeted_best_path if targeted_best_path.exists() else MODEL_DIR / 'jalon5_targeted_aug_ft_last.keras', compile=False)
combined_eval_model = tf.keras.models.load_model(combined_best_path if combined_best_path.exists() else MODEL_DIR / 'jalon5_combined_light_ft_last.keras', compile=False)

baseline_with_weak = evaluate_model(baseline_model, 'baseline_jalon4', weak_indices)
baseline_with_weak.to_csv(REPORT_DIR / 'baseline_metrics_with_weak.csv', index=False)

all_rows = []
all_rows.append(baseline_with_weak)
all_rows.append(evaluate_model(weighted_eval_model, 'weighted_loss', weak_indices))
all_rows.append(evaluate_model(targeted_eval_model, 'targeted_aug_oversampling', weak_indices))
all_rows.append(evaluate_model(combined_eval_model, 'combined_light', weak_indices))
comparison = pd.concat(all_rows, ignore_index=True)
comparison.to_csv(REPORT_DIR / 'jalon5_comparison_metrics.csv', index=False)
display(comparison)

orig_comparison = comparison[comparison['split'] == 'plantdoc_original'].copy()
orig_comparison = orig_comparison.sort_values(['macro_f1', 'weak_macro_f1', 'top1_accuracy'], ascending=False)
display(orig_comparison)

best_name = orig_comparison.iloc[0]['model']
if best_name == 'weighted_loss':
    best_src = weighted_best_path if weighted_best_path.exists() else MODEL_DIR / 'jalon5_weighted_loss_ft_last.keras'
elif best_name == 'targeted_aug_oversampling':
    best_src = targeted_best_path if targeted_best_path.exists() else MODEL_DIR / 'jalon5_targeted_aug_ft_last.keras'
elif best_name == 'combined_light':
    best_src = combined_best_path if combined_best_path.exists() else MODEL_DIR / 'jalon5_combined_light_ft_last.keras'
else:
    best_src = model_path

best_dst = MODEL_DIR / 'jalon5_weak_classes_best.keras'
if best_src != best_dst:
    shutil.copy2(best_src, best_dst)
print('Meilleur modèle selon PlantDoc original macro F1:', best_name)
print('Copié vers:', best_dst)

summary = {
    'start_model': str(model_path.relative_to(PROJECT_ROOT)),
    'best_model': str(best_dst.relative_to(PROJECT_ROOT)),
    'best_setting': str(best_name),
    'weak_classes': weak_df.to_dict(orient='records'),
    'comparison_csv': str((REPORT_DIR / 'jalon5_comparison_metrics.csv').relative_to(PROJECT_ROOT)),
}
(REPORT_DIR / 'jalon5_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False))


E0000 00:00:1781020777.572535   11349 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


,model,split,top1_accuracy,top2_accuracy,top3_accuracy,macro_f1,mean_confidence,n,weak_n,weak_top1_accuracy,weak_macro_f1
0,baseline_jalon4,plantdoc_original,0.516949,0.673729,0.775424,0.501717,0.578748,236,121,0.347107,0.212680
1,baseline_jalon4,plantdoc_segmented,0.508475,0.677966,0.758475,0.452295,0.552088,236,121,0.355372,0.220046
2,weighted_loss,plantdoc_original,0.516949,0.733051,0.838983,0.507769,0.594740,236,121,0.404959,0.246599
3,weighted_loss,plantdoc_segmented,0.538136,0.720339,0.800847,0.527878,0.569282,236,121,0.388430,0.267340
4,targeted_aug_oversampling,plantdoc_original,0.525424,0.724576,0.817797,0.529822,0.594008,236,121,0.429752,0.268677
5,targeted_aug_oversampling,plantdoc_segmented,0.550847,0.724576,0.805085,0.536078,0.566755,236,121,0.438017,0.323210
6,combined_light,plantdoc_original,0.495763,0.716102,0.809322,0.495837,0.584315,236,121,0.396694,0.245294
7,combined_light,plantdoc_segmented,0.546610,0.711864,0.796610,0.540829,0.558294,236,121,0.438017,0.326737


,model,split,top1_accuracy,top2_accuracy,top3_accuracy,macro_f1,mean_confidence,n,weak_n,weak_top1_accuracy,weak_macro_f1
4,targeted_aug_oversampling,plantdoc_original,0.525424,0.724576,0.817797,0.529822,0.594008,236,121,0.429752,0.268677
2,weighted_loss,plantdoc_original,0.516949,0.733051,0.838983,0.507769,0.594740,236,121,0.404959,0.246599
0,baseline_jalon4,plantdoc_original,0.516949,0.673729,0.775424,0.501717,0.578748,236,121,0.347107,0.212680
6,combined_light,plantdoc_original,0.495763,0.716102,0.809322,0.495837,0.584315,236,121,0.396694,0.245294


Meilleur modèle selon PlantDoc original macro F1: targeted_aug_oversampling
Copié vers: /home/leonard/UQAC/8INF934 - Atelier Pratique IA I/Projet/AgroScan/models/jalon5_weak_classes_best.keras


2910

## 15. Lecture des résultats

Le test Jalon 5 valide l'intérêt de cibler explicitement les classes faibles. Trois variantes ont été comparées à partir du modèle Jalon 4 exporté dans l'application :

1. pondération de loss (`weighted_loss`) ;
2. sur-échantillonnage + augmentation ciblée (`targeted_aug_oversampling`) ;
3. combinaison légère des deux (`combined_light`).

La meilleure variante globale reste **`targeted_aug_oversampling`**. La combinaison légère n'est pas retenue, car elle dégrade trop les performances sur PlantDoc original, qui est le jeu le plus proche du scénario mobile réel.

### Résumé chiffré sur PlantDoc original

| Modèle | Top-1 | Top-2 | Top-3 | Macro F1 | Weak top-1 | Weak macro F1 |
|---|---:|---:|---:|---:|---:|---:|
| Baseline Jalon 4 | 51,69 % | 67,37 % | 77,54 % | 50,17 % | 34,71 % | 21,27 % |
| Pondération de loss | 51,69 % | **73,31 %** | **83,90 %** | 50,78 % | 40,50 % | 24,66 % |
| Augmentation ciblée | **52,54 %** | 72,46 % | 81,78 % | **52,98 %** | **42,98 %** | **26,87 %** |
| Combinaison légère | 49,58 % | 71,61 % | 80,93 % | 49,58 % | 39,67 % | 24,53 % |

### Résumé chiffré sur PlantDoc segmenté

| Modèle | Top-1 | Top-2 | Top-3 | Macro F1 | Weak top-1 | Weak macro F1 |
|---|---:|---:|---:|---:|---:|---:|
| Baseline Jalon 4 | 50,85 % | 67,80 % | 75,85 % | 45,23 % | 35,54 % | 22,00 % |
| Pondération de loss | 53,81 % | 72,03 % | 80,08 % | 52,79 % | 38,84 % | 26,73 % |
| Augmentation ciblée | **55,08 %** | **72,46 %** | **80,51 %** | 53,61 % | **43,80 %** | 32,32 % |
| Combinaison légère | 54,66 % | 71,19 % | 79,66 % | **54,08 %** | **43,80 %** | **32,67 %** |

### Interprétation

La **pondération de loss** aide surtout le top-k. Sur PlantDoc original, le top-3 passe de **77,54 %** à **83,90 %**, ce qui signifie que la bonne classe est plus souvent présente parmi les premières hypothèses. En revanche, le top-1 reste identique à la baseline. Cette approche est donc intéressante pour l'interface top-3, mais elle ne suffit pas à améliorer clairement la prédiction principale.

Le **sur-échantillonnage avec augmentation ciblée** est le meilleur compromis. Il améliore le top-1, le macro F1 et les métriques sur classes faibles sur PlantDoc original. Le gain top-1 est modéré (**+0,85 point**), mais le macro F1 progresse plus nettement (**+2,81 points**) et le weak macro F1 passe de **21,27 %** à **26,87 %**. Sur PlantDoc segmenté, le gain est encore plus visible : top-1 **55,08 %** et macro F1 **53,61 %**.

La **combinaison légère** confirme qu'il ne faut pas simplement empiler les deux méthodes. Elle obtient le meilleur macro F1 et weak macro F1 sur PlantDoc segmenté, mais elle fait tomber PlantDoc original à **49,58 %** top-1 et **49,58 %** macro F1. Comme l'application reçoit d'abord des photos originales, cette dégradation est trop coûteuse. Le modèle combiné n'est donc pas retenu comme modèle final.

### Classes faibles ciblées

Les classes faibles retenues automatiquement sont principalement des maladies de tomate, pomme de terre, pomme, cerise, pêche et maïs. Les plus faibles au départ étaient notamment :

- `Tomato___Bacterial_spot` ;
- `Potato___Late_blight` ;
- `Tomato___Tomato_mosaic_virus` ;
- `Tomato___Leaf_Mold` ;
- `Peach___healthy` ;
- `Tomato___Early_blight` ;
- `Tomato___Septoria_leaf_spot`.

Ces classes ont peu d'images dans PlantDoc test, donc les métriques restent sensibles à quelques exemples. Les gains doivent être interprétés comme une amélioration expérimentale crédible, pas comme une validation agronomique définitive.

### Décision

Le meilleur modèle Jalon 5 reste :

```text
models/jalon5_weak_classes_best.keras
```

Il correspond à la variante **`targeted_aug_oversampling`**. C'est le modèle le plus pertinent à mentionner dans le rapport final pour la piste “renforcement des classes faibles”. La combinaison pondération + augmentation a été testée, mais elle n'est pas retenue car elle dégrade le comportement sur PlantDoc original.

Avant d'en faire le modèle mobile final, il reste pertinent de vérifier quelques exemples d'erreurs et, si le temps le permet, de comparer rapidement les CAM/Grad-CAM avec le modèle Jalon 4 pour s'assurer que le gain ne vient pas d'un biais visuel indésirable.
